# heatmap
> `CalendarHeatmap`: a Rich renderable GitHub-style annual heatmap.

In [ ]:
#| default_exp heatmap

In [ ]:
#| export
import math
from datetime import date, timedelta
from collections.abc import Mapping, Sequence
from rich.text import Text
from richcal.levels import quantile_thresholds, to_level
from richcal.layout import grid_bounds, year_blocks, month_label_cols

In [ ]:
#| export
_EMPTY = "grey23"
_GREEN_RAMP = ["#0e4429", "#006d32", "#26a641", "#39d353"]  # 4 activity shades

def default_palette(level_max):
    "Built-in GitHub-green palette of length level_max+1 (level 0 = empty)."
    if not 1 <= level_max <= len(_GREEN_RAMP):
        raise ValueError(f"no built-in palette for level_max={level_max}; pass palette=")
    return [_EMPTY] + _GREEN_RAMP[len(_GREEN_RAMP) - level_max:]

class CalendarHeatmap:
    "Rich renderable: GitHub-style annual activity heatmap from {date: value}."
    def __init__(self, data, start, end, level_max=4, thresholds=None,
                 palette=None, as_of=None, show_months=True,
                 show_weekdays=True, show_legend=True):
        if start > end: raise ValueError("start must be <= end")
        if level_max < 1: raise ValueError("level_max must be >= 1")
        if thresholds is not None:
            thresholds = list(thresholds)
            if len(thresholds) != level_max - 1:
                raise ValueError(f"thresholds length must be {level_max - 1}")
            if any(thresholds[i] > thresholds[i + 1] for i in range(len(thresholds) - 1)):
                raise ValueError("thresholds must be non-decreasing")
        observed = self._observed_values(data, start, end, as_of)
        for v in observed:
            if not math.isfinite(v) or v < 0:
                raise ValueError(f"observed values must be finite and >= 0; got {v!r}")
        self.data, self.start, self.end = data, start, end
        self.level_max, self.as_of = level_max, as_of
        self.show_months, self.show_weekdays, self.show_legend = show_months, show_weekdays, show_legend
        self.thresholds = thresholds if thresholds is not None else quantile_thresholds(observed, level_max)
        self.palette = list(palette) if palette is not None else default_palette(level_max)
        self.blocks = year_blocks(start, end)

    @staticmethod
    def _observed_values(data, start, end, as_of):
        "Values of in-window, non-future days that are present in data."
        out, d = [], start
        while d <= end:
            if (as_of is None or d <= as_of) and d in data:
                out.append(data[d])
            d += timedelta(days=1)
        return out

In [ ]:
from fastcore.test import test_eq, test_fail

# default palette length = level_max + 1
test_eq(len(default_palette(4)), 5)
test_eq(len(default_palette(3)), 4)
test_fail(lambda: default_palette(9), contains="palette")

# valid construction resolves thresholds and per-year blocks
hm = CalendarHeatmap({date(2026, 1, 1): 3, date(2026, 1, 2): 9},
                     date(2026, 1, 1), date(2026, 12, 31))
test_eq(len(hm.thresholds), 3)            # level_max-1
test_eq(len(hm.palette), 5)
test_eq([y for y, _, _ in hm.blocks], [2026])

# validation
test_fail(lambda: CalendarHeatmap({}, date(2026, 2, 1), date(2026, 1, 1)), contains="start")
test_fail(lambda: CalendarHeatmap({}, date(2026, 1, 1), date(2026, 12, 31), level_max=0), contains="level_max")
test_fail(lambda: CalendarHeatmap({}, date(2026, 1, 1), date(2026, 12, 31), thresholds=[1, 2]), contains="length")
test_fail(lambda: CalendarHeatmap({}, date(2026, 1, 1), date(2026, 12, 31), thresholds=[3, 2, 1]), contains="non-decreasing")
test_fail(lambda: CalendarHeatmap({date(2026, 1, 2): -1}, date(2026, 1, 1), date(2026, 12, 31)), contains="finite")

# a bad value on a FUTURE day must NOT raise (validation is observed-only)
CalendarHeatmap({date(2026, 12, 30): -1}, date(2026, 1, 1), date(2026, 12, 31), as_of=date(2026, 6, 23))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()